# Imports and install

In [1]:
# =========================
# Install (Colab)
# =========================
!pip uninstall -y google-generativeai -q
!pip install -q -U google-genai google-adk pydantic
!pip install "pymilvus>=2.4.0,<2.5"
!pip install faiss-cpu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 24.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which i

In [2]:
# =========================
# Imports
# =========================
from typing import TypeVar, Type, Union, Dict, List
import numpy as np

from pydantic import BaseModel
from google.colab import userdata

from google import genai
from google.adk import Agent
from google.adk.tools import google_search



# Dataset

In [3]:
import json
import requests
import os
import tempfile
import pandas as pd

FULL_DATASET_URL = "https://drive.google.com/file/d/1o3__6u1eR-VWAGgwOMacQCF96g4wKSfM/view?usp=sharing"

def download_file_from_google_drive(FULL_DATASET_URL):
  URL = "https://docs.google.com/uc?export=download"

  # Extract ID from google drive download link.
  file_id = FULL_DATASET_URL.split("/d/")[1].split("/")[0]
  download_url = f"https://drive.google.com/uc?export=download&id={file_id}"

  # Creates temp file
  temp_path = tempfile.mktemp(suffix=".json")

  # download
  response = requests.get(download_url)
  response.raise_for_status()
  with open(temp_path, "wb") as f:
      f.write(response.content)

  return temp_path

# Download the file
temp_path = download_file_from_google_drive(FULL_DATASET_URL)

# loads the JSON
with open(temp_path, "r") as f:
    candidates = json.load(f)

candidates_df = pd.DataFrame(candidates)


In [4]:
processed_candidates_dataset = 'https://drive.google.com/file/d/1TyRENYWxCU-kipJR4tDXLFRTbljm4_DR/view?usp=sharing'

temp_path = download_file_from_google_drive(processed_candidates_dataset)

processed_candidates = pd.read_csv(temp_path)

# processed_candidates['embedding_old'] = processed_candidates['embedding_old'].apply(eval)

#drop processed_candidates['embedding_old']
processed_candidates = processed_candidates.drop(columns=['embedding_old'])

processed_candidates['embedding'] = processed_candidates['embedding'].apply(eval)

#convert full_experience to dict
processed_candidates['full_experience'] = processed_candidates['full_experience'].apply(eval)




# Gen AI

In [7]:
# =========================
# Generic typed LLM caller
# =========================
from typing import TypeVar
from typing import Optional
from pydantic import BaseModel

T = TypeVar("T")

MODEL_NAME = "gemini-2.5-flash"


class WorkExperience(BaseModel):
    company: str
    job_title: str
    details: str
    start_date: str
    end_date: Optional[str]


ResumeExperiences = list[WorkExperience]


def call_llm(
    *,
    system_prompt: str,
    user_prompt: str,
    output_type: type[T],
    model: str = MODEL_NAME,
    temperature: float = 0.0,
) -> T:

    response = google_client.models.generate_content(
        model=model,

        contents=user_prompt,

        config={
            "system_instruction": system_prompt,
            "temperature": temperature,

            "response_mime_type": "application/json",
            "response_schema": output_type,
        },
    )

    return response.parsed

In [8]:
# skills = call_llm(
#     system_prompt="""
#     Extract technical skills only.
#     Return unique values.
#     """,

#     user_prompt="""
#     Candidate knows Python, Spark, AWS and Python.
#     """,

#     output_type=list[str],
# )

# print(skills)

In [9]:
# resume_content = candidates_df['content'].iloc[0]
# resume_content

# experiences = call_llm(
#     system_prompt=RESUME_PARSER_SYSTEM_PROMPT,
#     user_prompt=resume_content,
#     output_type=ResumeExperiences,
# )

# parsed = [
#     exp.model_dump()
#     for exp in experiences
# ]
# parsed

# Agent

In [10]:
# =========================
# ADK Agent
# =========================
research_agent = Agent(
    name="researcher",
    model=MODEL_NAME,
    instruction="""
    You help users research topics thoroughly.
    Always use tools when external information is needed.
    """,
    tools=[google_search],
)

In [11]:
def embed(text: str) -> list[float]:
    response = google_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    return response.embeddings[0].values

def embed_batch(texts: list[str]) -> list[list[float]]:
    response = google_client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts
    )
    return [e.values for e in response.embeddings]

# Knowledge base

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.3 MB/s eta 0:00:00


In [21]:
from faiss_store import FAISSVectorStore
from schemas import CANDIDATE_SCHEMA,JOB_SCHEMA
from milvus_store import MilvusVectorStore
from candidate_retriever import CandidateRetriever



In [12]:
# FAISS
#faiss_store = FAISSVectorStore()

#Milvus
store = MilvusVectorStore()



In [22]:
store.create_index(
    index_name="candidates_index",
    schema=CANDIDATE_SCHEMA,
)


store.create_index(
    index_name="jobs_index",
    schema=JOB_SCHEMA,
)


retriever = CandidateRetriever(
    vector_store=store,
)

In [23]:
candidates_to_index = processed_candidates.copy()

from vector_store import build_records


records = build_records(
    rows=candidates_to_index.to_dict(
        orient="records"
    ),

    id_field="candidate_id",
)


store.insert(
    index_name="candidates_index",
    records=records,
)

In [24]:
retriever.search("INFORMATION TECHNOLOGY TECHNICIAN")

[{'candidate_id': '2c9a6530-5a2e-42f8-8776-0132d4bc9c99',
  'first_name': 'Cameron',
  'last_name': 'Miller',
  'content': 'INFORMATION TECHNOLOGY TECHNICIAN I       Summary     Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network security. Experie...',
  'score': 0.693},
 {'candidate_id': 'e6920061-624e-4a91-802b-802695dfea5c',
  'first_name': 'Sam',
  'last_name': 'Davis',
  'content': 'INFORMATION TECHNOLOGY MANAGER       Professional Summary    Possesses an extensive background in Information Technology Management, along with a Masters of Science degree and multiple certifications....',
  'score': 0.64},
 {'candidate_id': 'ebb0030d-6b5d-4cb2-a410-efdca8293e4d',
  'first_name': 'Jordan',
  'last_name': 'Clark',
  'content': 'AVIATION ELECTRONICS TECHNICIAN       Summary    Seeking a challenging position where my existing skills and experience contribute to your company while expanding my technical abilities.

# Google Cloud

### 1. Configurar o Google Cloud Project e inicializar o Vertex AI

Primeiro, precisamos importar as bibliotecas necessárias e configurar o seu projeto Google Cloud. **Lembre-se de preencher `YOUR_PROJECT_ID`, `YOUR_GCS_BUCKET_NAME` e `YOUR_GCP_REGION`**.

In [ ]:
import json
import pandas as pd
from google.cloud import aiplatform, storage
import tempfile
import os

# --- Configurações do seu Google Cloud Project ---
# TODO: Substitua pelos seus valores reais
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"  # Exemplo: 'us-central1', 'southamerica-east1'
GCS_BUCKET_NAME = "YOUR_GCS_BUCKET_NAME" # Um bucket GCS para armazenar os dados do índice

# Inicializa o cliente do Vertex AI
aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI inicializado para o projeto: {PROJECT_ID} na região: {LOCATION}")

### 2. Preparar os dados para o Vector Search

Converteremos o `processed_df` para o formato JSONL exigido pelo Vertex AI Vector Search. Cada linha conterá o `id` do candidato, o `embedding` de 3072 dimensões e as informações adicionais como `first_name`, `last_name`, `content` e `full_experience` aninhadas sob uma chave `metadata`.

In [ ]:
candidate_data_for_vector_search = []
for _, row in processed_df.iterrows():
    candidate_data_for_vector_search.append({
        "id": str(row['id']),  # Garante que o ID é uma string
        "embedding": row['embedding'],
        "metadata": {
            "first_name": row['first_name'],
            "last_name": row['last_name'],
            "content": row['content'],
            "full_experience": row['full_experience']
        }
    })

# --- Salvar dados em um arquivo temporário local ---
local_temp_file_path = os.path.join(tempfile.gettempdir(), "candidate_embeddings.jsonl")
with open(local_temp_file_path, "w") as f:
    for item in candidate_data_for_vector_search:
        f.write(json.dumps(item) + "\n")

print(f"Dados preparados e salvos em arquivo local temporário: {local_temp_file_path}")

### 3. Fazer upload dos dados para o Google Cloud Storage (GCS)

O Vertex AI Vector Search requer que os dados de origem sejam armazenados em um bucket GCS. Este passo faz o upload do arquivo JSONL que acabamos de criar.

In [ ]:
# Garante que o bucket GCS exista e que você tenha permissões adequadas.
# Se o bucket não existir, você pode criá-lo via console do GCP ou com o código:
# storage_client.create_bucket(GCS_BUCKET_NAME)
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET_NAME)

gcs_blob_path = f"vector_search_data/candidate_embeddings_{os.path.basename(local_temp_file_path)}"
blob = bucket.blob(gcs_blob_path)
blob.upload_from_filename(local_temp_file_path)

gcs_uri = f"gs://{GCS_BUCKET_NAME}/{gcs_blob_path}"
print(f"Dados enviados para o GCS: {gcs_uri}")

### 4. Criar o Índice do Vertex AI Vector Search

Agora usaremos o URI do GCS para criar o índice. Este processo pode levar algum tempo para ser concluído, dependendo do volume de dados.

In [ ]:
INDEX_DISPLAY_NAME = "candidate_resume_index"
INDEX_DESCRIPTION = "Vector Search Index para currículos de candidatos com metadados"
EMBEDDING_DIMENSIONS = 3072  # Conforme especificado pelo usuário
DISTANCE_MEASURE_TYPE = "DOT_PRODUCT" # Ou "COSINE_DISTANCE", "L2_DISTANCE"
FEATURE_NORM_TYPE = "UNIT_L2_NORM" # Ou "NONE"

# Cria o objeto Index
# Este processo pode levar tempo, você pode monitorá-lo no console do Google Cloud.
my_index = aiplatform.Index.create_from_gcs(
    display_name=INDEX_DISPLAY_NAME,
    contents_delta_uri=gcs_uri,
    is_partial_update=False, # Defina como True se estiver atualizando um índice existente
    metadata=aiplatform.MatchingEngineIndex.create_from_gcs_metadata.MatchingEngineIndexConfig(
        dimensions=EMBEDDING_DIMENSIONS,
        distance_measure_type=DISTANCE_MEASURE_TYPE,
        feature_norm_type=FEATURE_NORM_TYPE,
        # Configurações de sharding e tipo de máquina
        tree_ah_config=aiplatform.MatchingEngineIndex.create_from_gcs_metadata.MatchingEngineIndexConfig.TreeAhConfig(
            leaf_node_embedding_count=500,
            leaf_node_emb_group_count=1,
        )
    ),
    description=INDEX_DESCRIPTION,
    project=PROJECT_ID,
    location=LOCATION
)

print(f"O processo de criação do índice foi iniciado. Nome do recurso do índice: {my_index.resource_name}")
print("Você pode monitorar o status do índice no console do Google Cloud em Vertex AI > Vector Search > Indexes.")

### 5. Implantar o Índice (Próximo Passo)

Após a criação bem-sucedida do índice (o que pode levar algumas dezenas de minutos a horas), o próximo passo é implantá-lo em um `IndexEndpoint` para que ele possa ser consultado. Este processo também pode levar tempo.

Para implantar o índice, você precisaria de um código semelhante ao abaixo (comentado aqui porque a criação do índice ainda estará em andamento):

```python
# DEPLOYED_INDEX_ID = "candidate_resume_deployed_index" # Deve ser único

# my_index_endpoint = aiplatform.IndexEndpoint.create(
#     display_name=f"{INDEX_DISPLAY_NAME}_endpoint",
#     project=PROJECT_ID,
#     location=LOCATION
# )

# my_index_endpoint.deploy_index(
#     index=my_index,
#     deployed_index_id=DEPLOYED_INDEX_ID,
#     machine_type="e2-highmem-2", # Ou um tipo de máquina que se adapte às suas necessidades
#     min_replica_count=1,
#     max_replica_count=1
# )

# print(f"Índice implantado no endpoint: {my_index_endpoint.resource_name}")
# print(f"ID do índice implantado: {DEPLOYED_INDEX_ID}")
```